In [ ]:
import os
import getpass

from langchain.document_loaders import PyPDFDirectoryLoader, CSVLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.chat_models import ChatOpenAI
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

In [ ]:
# Cria um campo seguro para você colar a sua API Key
os.environ["OPENAI_API_KEY"] = getpass.getpass("Cole sua chave da OpenAI (sk-...): ")

In [ ]:
# =====================================================================
# SKILL 1: ESPECIALISTA EM METADADOS (Portfólio Consolidado)
# =====================================================================
def criar_skill_metadados():
    print("Carregando Portfólio de Metadados (CSV)...")
    loader_csv = CSVLoader("portifolio/portfolio_metadados.csv")
    documentos_csv = loader_csv.load()
    
    embeddings = OpenAIEmbeddings()
    vectorstore_csv = FAISS.from_documents(documentos_csv, embeddings)
    retriever_csv = vectorstore_csv.as_retriever(search_kwargs={"k": 2})
    
    llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)
    return RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever_csv)

# =====================================================================
# SKILL 2: ESPECIALISTA EM CONTRATOS (Leitura de PDF)
# =====================================================================
def criar_skill_contratos():
    print("Carregando Contratos Jurídicos (PDF)...")
    loader_pdf = PyPDFDirectoryLoader("contratos/")
    documentos_pdf = loader_pdf.load()
    
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    textos_divididos = text_splitter.split_documents(documentos_pdf)
    
    embeddings = OpenAIEmbeddings()
    vectorstore_pdf = FAISS.from_documents(textos_divididos, embeddings)
    retriever_pdf = vectorstore_pdf.as_retriever(search_kwargs={"k": 3})
    
    llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)
    return RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever_pdf)

# =====================================================================
# AGENTE MASTER (Roteador Inteligente)
# =====================================================================
def agente_master_roteador(pergunta, skill_metadados, skill_contratos):
    llm_master = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)
    
    # O Cérebro do Master: Decide qual especialista chamar
    prompt_roteamento = f"""
    Você é o Agente Orquestrador (Master). Analise a pergunta do usuário e decida qual especialista chamar.
    
    Opções:
    1. "METADADOS" - Se a pergunta for sobre portfólio, quais tabelas existem, descrição de produtos ou status de contratação (ativo/inativo).
    2. "CONTRATOS" - Se a pergunta for sobre regras jurídicas, custos, preços, vigência ou limites de chamadas de API.
    
    Pergunta: {pergunta}
    
    Responda APENAS com a palavra "METADADOS" ou "CONTRATOS".
    """
    
    # Manda a pergunta pro Master decidir
    decisao = llm_master.predict(prompt_roteamento).strip().upper()
    
    print(f"🧠 [Agente Master] Intenção identificada. Delegando para o Especialista em: {decisao}")
    
    # Executa a Skill correta baseada na decisão
    if "METADADOS" in decisao:
        return skill_metadados.run(pergunta)
    else:
        return skill_contratos.run(pergunta)

# --- Simulação da Interação no Jupyter ---
if __name__ == "__main__":
    print("\nInicializando bases de conhecimento...")
    skill_meta = criar_skill_metadados()
    skill_cont = criar_skill_contratos()
    
    print("\n✅ Agente Master Orquestrador Ativo. Aguardando intenções...\n")
    
    # Teste 1: Roteamento para o Especialista em Metadados
    print("\n--- TESTE 1 ---")
    pergunta_1 = "Temos alguma tabela do BACEN contratada? Qual é a descrição dela?"
    print(f"👤 Utilizador: {pergunta_1}")
    resposta_1 = agente_master_roteador(pergunta_1, skill_meta, skill_cont)
    print(f"🤖 Resposta: {resposta_1}")
    
    # Teste 2: Roteamento para o Especialista em Contratos
    print("\n--- TESTE 2 ---")
    pergunta_2 = "Preciso saber qual é o custo unitário por requisição da API do BACEN."
    print(f"👤 Utilizador: {pergunta_2}")
    resposta_2 = agente_master_roteador(pergunta_2, skill_meta, skill_cont)
    print(f"🤖 Resposta: {resposta_2}")